# Notebook 10 — Stacking Ensemble

**Combina las puntuaciones de los detectores base** (NB03 IF, NB03b K-Means+IF, NB04 Dense-AE, NB05 LSTM-AE, NB07 CNN-AE, NB08 Transformer-AE) **mediante un meta-clasificador**. Aplica directamente la estrategia de *stacking sobre Out-Of-Fold* del **Proyecto 3 de NLP**.

**¿Por qué stacking funciona aquí?** Los seis detectores tienen *errores complementarios*: IF es excelente en saltos abruptos (`step`); Dense-AE captura mejor las anomalías de magnitud sostenidas (`scaling`, `offset`); LSTM-AE detecta antes los `replay` y `ramp` por la memoria temporal; el Transformer aporta atención global; CNN-AE captura patrones temporales locales. Un meta-aprendizaje sobre sus salidas extrae lo mejor de cada uno.

**Diseño honesto sin leakage.**
- Cada detector base se entrenó en datos limpios de **train** y se calibró en **val**.
- Para entrenar el meta-LR usamos los scores de los detectores sobre **val** (donde hay ataques con ground truth pero los detectores no han visto las etiquetas durante el entrenamiento).
- Evaluamos el ensemble final sobre **test** (que ningún detector ni el meta-LR han visto durante el entrenamiento de pesos).

**Estrategias incluidas:**
1. **Mean blending** (promedio simple, sin pesos aprendidos) — baseline.
2. **Weighted blending** con pesos óptimos en val (búsqueda en grid).
3. **Meta-LogisticRegression** sobre las 6 puntuaciones (puede aprender no-monotonías).
4. **Meta-LightGBM** (no-lineal sobre los scores).

Comparación final por AUC-ROC, AUC-PR, F1, latencia y tamaño total del ensemble.


## 0. Setup

In [4]:
import os, json, time, warnings, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 12
print('OK')

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
    f1_score, fbeta_score, precision_score, recall_score,
    average_precision_score, precision_recall_curve
)
import joblib
try:
    import lightgbm as lgb
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False
print('LightGBM disponible:', HAS_LGBM)


OK
LightGBM disponible: True


## 1. Carga de Datos Reales + Configuración

In [5]:
BASE_DIR = '/Volumes/Extreme Pro Particion 1TB/TFG/UCIrvine/'
DATA_DIR = os.path.join(BASE_DIR, 'data')

with open(os.path.join(DATA_DIR, 'pipeline_config.json')) as f:
    config = json.load(f)

FEATURE_COLS = config['feature_cols']
ATTACK_TYPES = config['attack_types']

df_train = pd.read_csv(os.path.join(DATA_DIR, 'train_clean.csv'),
                       index_col='datetime', parse_dates=True)
df_val   = pd.read_csv(os.path.join(DATA_DIR, 'val_with_attacks.csv'),
                       index_col='datetime', parse_dates=True)
df_test  = pd.read_csv(os.path.join(DATA_DIR, 'test_with_attacks.csv'),
                       index_col='datetime', parse_dates=True)

print('DATOS CARGADOS (UCI real)')
print('=' * 65)
for nm, d in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    pos = (d['label']==1).sum() if 'label' in d.columns else 0
    print(f'  {nm:6s} {len(d):>9,d}   ataques: {pos:>7,d}')

y_val  = df_val['label'].values
y_test = df_test['label'].values
print(f'\nVal: {len(y_val):,} (pos {(y_val==1).sum():,})')
print(f'Test: {len(y_test):,} (pos {(y_test==1).sum():,})')


DATOS CARGADOS (UCI real)
  Train  1,300,061   ataques:       0
  Val      144,452   ataques:  21,739
  Test     604,999   ataques:  96,717

Val: 144,452 (pos 21,739)
Test: 604,999 (pos 96,717)


## 2. Carga de scores de los detectores base

In [6]:
# Cargamos predictions_*.csv de cada detector. Cada notebook usa un nombre
# distinto para la columna de score (p.ej. if_score, dae_score, lstm_score,
# kmif_score, o simplemente 'score' en CNN-AE / Transformer-AE).
# Aqui detectamos automaticamente la columna correcta.
#
# Mapa de detectores. Si alguno no existe todavia, se ignora con un warning.
DETECTORS = {
    'if':           'predictions_isolation_forest.csv',
    'kmeans_if':    'predictions_kmeans_if.csv',
    'dense_ae':     'predictions_dense_autoencoder.csv',
    'lstm_ae':      'predictions_lstm_autoencoder.csv',
    'cnn_ae':       'predictions_cnn_autoencoder.csv',
    'transformer':  'predictions_transformer_autoencoder.csv',
}

def pick_score_column(df):
    '''Devuelve el nombre de la columna de score. Acepta 'score' o '*_score'
    (excluyendo columnas que contengan 'pred').'''
    if 'score' in df.columns:
        return 'score'
    cands = [c for c in df.columns
             if c.endswith('_score') and 'pred' not in c]
    if not cands:
        raise KeyError(f"No score column found. Columnas: {list(df.columns)}")
    return cands[0]

# Cargar scores en test
scores_test_d = {}
present = []
for name, fname in DETECTORS.items():
    path = os.path.join(DATA_DIR, fname)
    if not os.path.exists(path):
        print(f'  ! {name:14s} (no existe {fname}) -> omitido')
        continue
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    score_col = pick_score_column(df)
    scores_test_d[name] = df[score_col].values
    present.append(name)
    print(f'  > {name:14s} {df.shape}   score_col={score_col}')

print(f'\nDetectores disponibles en test: {present}')


  > if             (604999, 16)   score_col=if_score
  > kmeans_if      (604999, 17)   score_col=kmif_score
  > dense_ae       (604999, 16)   score_col=dae_score
  > lstm_ae        (604999, 16)   score_col=lstm_score
  > cnn_ae         (604999, 6)   score_col=score
  > transformer    (604999, 6)   score_col=score

Detectores disponibles en test: ['if', 'kmeans_if', 'dense_ae', 'lstm_ae', 'cnn_ae', 'transformer']


## 3. Generar scores en val para los modelos base (one-off)

In [ ]:
# Necesitamos scores_val_d[name] para entrenar el meta-LR.
# Si tus notebooks NB03..NB08 ya guardaron predictions_*_val.csv, los cargamos;
# de lo contrario, los regeneramos aqui cargando los modelos guardados.

import joblib
from tensorflow.keras.models import load_model
import tensorflow as tf
try:
    tf.keras.mixed_precision.set_global_policy('float32')
except Exception:
    pass

def load_pipeline_config_for(name):
    '''Carga scaler, threshold y modelo segun nombre.'''
    paths = {
        'if': ('scaler_isolation_forest.pkl', 'model_isolation_forest.pkl', None, 'sklearn'),
        'kmeans_if': ('scaler_kmeans_if.pkl', 'model_kmeans_if.pkl', None, 'kmeans_if'),
        'dense_ae': ('scaler_dense_autoencoder.pkl', 'model_dense_autoencoder.keras', 'feature_weights_dense_ae.npy', 'keras_ae'),
        'lstm_ae':  ('scaler_lstm_autoencoder.pkl',  'model_lstm_autoencoder.keras',  'feature_weights_lstm_ae.npy', 'keras_lstm'),
        'cnn_ae':   ('scaler_cnn_autoencoder.pkl',   'model_cnn_autoencoder.keras',   'feature_weights_cnn_ae.npy',  'keras_cnn'),
        'transformer': ('scaler_transformer_autoencoder.pkl', 'model_transformer_autoencoder.keras', 'patch_weights_transformer_ae.npy', 'keras_tr'),
    }
    return paths.get(name)


def compute_features(df, mode='full'):
    f = df[FEATURE_COLS].copy()
    f['VI_residual'] = (df['Global_active_power']
                        - df['Voltage'] * df['Global_intensity'] / 1000.0)
    if mode in ('medium','full'):
        h = df.index.hour + df.index.minute / 60.0
        f['hour_sin'] = np.sin(2*np.pi*h/24); f['hour_cos'] = np.cos(2*np.pi*h/24)
        d = df.index.dayofweek
        f['dow_sin']  = np.sin(2*np.pi*d/7);  f['dow_cos']  = np.cos(2*np.pi*d/7)
        f['gap_diff1'] = df['Global_active_power'].diff().fillna(0)
    if mode == 'full':
        f['vi_res_abs'] = f['VI_residual'].abs()
        f['vi_res_roll15_mean'] = f['vi_res_abs'].rolling(15, min_periods=1).mean()
        f['gap_intensity_ratio'] = df['Global_active_power'] / (df['Global_intensity']+0.01)
        f['sm_gap_ratio'] = ((df['Sub_metering_1']+df['Sub_metering_2']+df['Sub_metering_3'])/1000.0
                              / (df['Global_active_power']+0.01))
    return f.fillna(0)


def score_dense_ae(model, scaler, weights, df):
    # iter NB10: Dense-AE entrenado con mode='full' (17 features)
    Xf = compute_features(df, mode='full')
    X = scaler.transform(Xf.values).astype('float32')
    rec = model.predict(X, batch_size=2048, verbose=0)
    return ((rec - X)**2 * weights).sum(axis=1)


def make_windows(X, window=60, stride=1):
    n = len(X)
    if n < window:
        return np.empty((0, window, X.shape[1]), dtype='float32')
    starts = np.arange(0, n - window + 1, stride)
    return np.stack([X[s:s+window] for s in starts]).astype('float32')


def expand(scores_w, n_total, window=60):
    out = np.full(n_total, np.nan, dtype='float32')
    for s, sc in enumerate(scores_w):
        out[s:s+window] = np.maximum(np.nan_to_num(out[s:s+window], nan=-np.inf), sc)
    nan_mask = np.isnan(out)
    if nan_mask.any():
        first_valid = np.argmax(~nan_mask)
        out[nan_mask] = out[first_valid]
    return out


def score_lstm_ae(model, scaler, weights, df, window=60):
    # iter NB10: LSTM-AE entrenado con mode='full' (17 features)
    Xf = compute_features(df, mode='full')
    X = scaler.transform(Xf.values).astype('float32')
    Xw = make_windows(X, window=window, stride=1)
    if len(Xw) == 0:
        return np.zeros(len(df), dtype='float32')
    rec = model.predict(Xw, batch_size=512, verbose=0)
    err = ((rec - Xw)**2).mean(axis=1)             # (N, F)
    sc_w = (err * weights).sum(axis=1)             # (N,)
    return expand(sc_w, len(df), window=window)


def score_cnn_ae(model, scaler, weights, df, window=60):
    # CNN-AE entrenado con mode='medium'+vi_res_abs (14 features)
    Xf = compute_features(df, mode='medium')
    Xf['vi_res_abs'] = Xf['VI_residual'].abs()
    X = scaler.transform(Xf.values).astype('float32')
    Xw = make_windows(X, window=window, stride=1)
    if len(Xw) == 0:
        return np.zeros(len(df), dtype='float32')
    rec = model.predict(Xw, batch_size=512, verbose=0)
    err = ((rec - Xw)**2).mean(axis=1)
    sc_w = (err * weights).sum(axis=1)
    return expand(sc_w, len(df), window=window)


def score_transformer(model, scaler, patch_weights, df, window=60, patch_len=5):
    Xf = compute_features(df, mode='medium')
    Xf['vi_res_abs'] = Xf['VI_residual'].abs()
    X = scaler.transform(Xf.values).astype('float32')
    Xw = make_windows(X, window=window, stride=1)
    if len(Xw) == 0:
        return np.zeros(len(df), dtype='float32')
    N, W, F = Xw.shape
    Xp = Xw.reshape(N, W//patch_len, patch_len, F).reshape(N, W//patch_len, patch_len*F)
    rec = model.predict(Xp, batch_size=512, verbose=0)
    err = ((rec - Xp)**2).mean(axis=2)             # (N, n_patches)
    sc_w = (err * patch_weights).sum(axis=1)
    return expand(sc_w, len(df), window=window)


def score_if(model, scaler, df):
    Xf = compute_features(df, mode='full')
    X = scaler.transform(Xf.values)
    # IF: -score_samples ya es "anomalia"
    return -model.score_samples(X)


# Calcular scores en val (memoria-amistoso: uno por uno)
scores_val_d = {}
for name in present:
    try:
        cfg = load_pipeline_config_for(name)
        if cfg is None:
            continue
        sc_fn, mdl_fn, w_fn, kind = cfg
        scaler = joblib.load(os.path.join(DATA_DIR, sc_fn))
        weights = np.load(os.path.join(DATA_DIR, w_fn)) if w_fn else None
        if kind == 'sklearn':
            mdl = joblib.load(os.path.join(DATA_DIR, mdl_fn))
            sc = score_if(mdl, scaler, df_val)
        elif kind == 'kmeans_if':
            # K-Means+IF tiene su propia logica; cargamos el modelo y dejamos
            # que use su funcion interna. Si no esta disponible, omitimos.
            try:
                mdl = joblib.load(os.path.join(DATA_DIR, mdl_fn))
                sc = score_if(mdl, scaler, df_val)
            except Exception as e:
                print(f'  ! {name}: {e}'); continue
        elif kind == 'keras_ae':
            mdl = load_model(os.path.join(DATA_DIR, mdl_fn))
            sc = score_dense_ae(mdl, scaler, weights, df_val)
        elif kind == 'keras_lstm':
            mdl = load_model(os.path.join(DATA_DIR, mdl_fn))
            sc = score_lstm_ae(mdl, scaler, weights, df_val)
        elif kind == 'keras_cnn':
            mdl = load_model(os.path.join(DATA_DIR, mdl_fn))
            sc = score_cnn_ae(mdl, scaler, weights, df_val)
        elif kind == 'keras_tr':
            mdl = load_model(os.path.join(DATA_DIR, mdl_fn))
            sc = score_transformer(mdl, scaler, weights, df_val)
        else:
            continue
        scores_val_d[name] = np.asarray(sc, dtype='float32')
        print(f'  > {name:14s} val scores: {scores_val_d[name].shape}')
    except Exception as e:
        print(f'  ! {name}: {e}')


## 4. Normalización de scores

In [8]:
# Cada detector tiene un rango de score distinto.
# Estandarizamos por z-score usando estadisticos de val (negativos limpios).

def standardize(scores, ref):
    mu = np.mean(ref); sd = np.std(ref) + 1e-9
    return (scores - mu) / sd

models_in_stack = [n for n in present if n in scores_val_d and n in scores_test_d]
print(f'Modelos finales en el stack ({len(models_in_stack)}):', models_in_stack)

S_val  = np.column_stack([standardize(scores_val_d[n],  scores_val_d[n][y_val==0]) for n in models_in_stack])
S_test = np.column_stack([standardize(scores_test_d[n], scores_val_d[n][y_val==0]) for n in models_in_stack])
print(f'S_val: {S_val.shape}  S_test: {S_test.shape}')


Modelos finales en el stack (3): ['if', 'cnn_ae', 'transformer']
S_val: (144452, 3)  S_test: (604999, 3)


## 5. Estrategia 1: Mean Blending

In [9]:
def calibrate_threshold(y_true, scores, beta=1.0):
    '''Busca umbral que maximiza F_beta usando precision_recall_curve.'''
    precisions, recalls, thresholds = precision_recall_curve(y_true, scores)
    precisions = precisions[:-1]
    recalls    = recalls[:-1]
    fbeta = ((1 + beta**2) * precisions * recalls /
             (beta**2 * precisions + recalls + 1e-10))
    best_idx = np.argmax(fbeta)
    best_thr = thresholds[best_idx]
    return best_thr, precisions[best_idx], recalls[best_idx], fbeta[best_idx]

s_val_mean  = S_val.mean(axis=1)
s_test_mean = S_test.mean(axis=1)
thr_m, _, _, _ = calibrate_threshold(y_val, s_val_mean)
y_pred_m = (s_test_mean > thr_m).astype(int)
auc_roc_m = roc_auc_score(y_test, s_test_mean)
auc_pr_m  = average_precision_score(y_test, s_test_mean)
f1_m  = f1_score(y_test, y_pred_m)
print(f'Mean Blending: AUC-ROC={auc_roc_m:.4f}  AUC-PR={auc_pr_m:.4f}  F1={f1_m:.4f}')


Mean Blending: AUC-ROC=0.8901  AUC-PR=0.5938  F1=0.6354


## 6. Estrategia 2: Weighted Blending (busqueda en grid)

In [10]:
from itertools import product

def grid_blend(S_val, y_val, S_test, n_steps=4):
    '''Busqueda en grid de pesos discretos sobre simplex.'''
    n_models = S_val.shape[1]
    # Pesos discretos {0, 0.33, 0.67, 1.0}
    grid_w = np.linspace(0, 1, n_steps+1)
    best = {'auc_pr': 0, 'weights': None, 'thr': 0.5}
    combos = 0
    for ws in product(grid_w, repeat=n_models):
        s = sum(ws)
        if s == 0: continue
        ws_norm = np.array(ws) / s
        s_val_w = S_val @ ws_norm
        # Calibracion fina del threshold para esta combinacion
        try:
            thr, _, _, _ = calibrate_threshold(y_val, s_val_w)
        except Exception:
            continue
        ap = average_precision_score(y_val, s_val_w)
        if ap > best['auc_pr']:
            best.update(auc_pr=ap, weights=ws_norm, thr=thr)
        combos += 1
    return best

t0 = time.time()
best_blend = grid_blend(S_val, y_val, S_test, n_steps=4)
print(f'Busqueda en {time.time()-t0:.1f}s')
print(f'Pesos optimos:')
for n, w in zip(models_in_stack, best_blend["weights"]):
    print(f'   {n:14s} {w:.3f}')
print(f'  thr: {best_blend["thr"]:.6f}')

s_test_wb = S_test @ best_blend['weights']
y_pred_wb = (s_test_wb > best_blend['thr']).astype(int)
auc_roc_wb = roc_auc_score(y_test, s_test_wb)
auc_pr_wb  = average_precision_score(y_test, s_test_wb)
f1_wb = f1_score(y_test, y_pred_wb)
print(f'Weighted Blending: AUC-ROC={auc_roc_wb:.4f}  AUC-PR={auc_pr_wb:.4f}  F1={f1_wb:.4f}')


Busqueda en 2.8s
Pesos optimos:
   if             0.800
   cnn_ae         0.000
   transformer    0.200
  thr: 1.631508
Weighted Blending: AUC-ROC=0.9274  AUC-PR=0.7636  F1=0.7317


## 7. Estrategia 3: Meta-Logistic Regression

In [11]:
meta_lr = LogisticRegression(C=2.0, max_iter=2000, class_weight='balanced',
                              solver='liblinear', random_state=42)
meta_lr.fit(S_val, y_val)
s_test_lr = meta_lr.predict_proba(S_test)[:, 1]
thr_lr, _, _, _ = calibrate_threshold(y_val, meta_lr.predict_proba(S_val)[:, 1])
y_pred_lr = (s_test_lr > thr_lr).astype(int)
auc_roc_lr = roc_auc_score(y_test, s_test_lr)
auc_pr_lr  = average_precision_score(y_test, s_test_lr)
f1_lr = f1_score(y_test, y_pred_lr)
print(f'Meta-LR:  AUC-ROC={auc_roc_lr:.4f}  AUC-PR={auc_pr_lr:.4f}  F1={f1_lr:.4f}')
print(f'  Pesos (coef): {dict(zip(models_in_stack, meta_lr.coef_[0].round(3)))}')
print(f'  Intercept: {meta_lr.intercept_[0]:.3f}')


Meta-LR:  AUC-ROC=0.9339  AUC-PR=0.7923  F1=0.7556
  Pesos (coef): {'if': np.float64(1.391), 'cnn_ae': np.float64(-0.391), 'transformer': np.float64(0.385)}
  Intercept: -1.606


## 8. Estrategia 4: Meta-LightGBM (no lineal)

In [12]:
if HAS_LGBM:
    dtrain = lgb.Dataset(S_val, label=y_val)
    params = dict(objective='binary', metric='auc', learning_rate=0.05,
                  num_leaves=15, max_depth=4, min_data_in_leaf=100,
                  is_unbalance=True, num_threads=10, verbose=-1, seed=42)
    gbm_meta = lgb.train(params, dtrain, num_boost_round=200,
                         valid_sets=[dtrain],
                         callbacks=[lgb.log_evaluation(period=0)])
    s_test_gbm = gbm_meta.predict(S_test)
    thr_gbm, _, _, _ = calibrate_threshold(y_val, gbm_meta.predict(S_val))
    y_pred_gbm = (s_test_gbm > thr_gbm).astype(int)
    auc_roc_gbm = roc_auc_score(y_test, s_test_gbm)
    auc_pr_gbm  = average_precision_score(y_test, s_test_gbm)
    f1_gbm = f1_score(y_test, y_pred_gbm)
    print(f'Meta-LGBM: AUC-ROC={auc_roc_gbm:.4f}  AUC-PR={auc_pr_gbm:.4f}  F1={f1_gbm:.4f}')
else:
    print('LightGBM no disponible — saltado.')
    auc_roc_gbm = auc_pr_gbm = f1_gbm = None
    s_test_gbm = None; thr_gbm = None


Meta-LGBM: AUC-ROC=0.9710  AUC-PR=0.9204  F1=0.8422


## 9. Comparación + selección del MEJOR ensemble

In [13]:
summary = pd.DataFrame([
    {'estrategia':'Mean blending',     'AUC-ROC':auc_roc_m,  'AUC-PR':auc_pr_m,  'F1':f1_m},
    {'estrategia':'Weighted blending', 'AUC-ROC':auc_roc_wb, 'AUC-PR':auc_pr_wb, 'F1':f1_wb},
    {'estrategia':'Meta-LR',           'AUC-ROC':auc_roc_lr, 'AUC-PR':auc_pr_lr, 'F1':f1_lr},
])
if HAS_LGBM:
    summary = pd.concat([summary, pd.DataFrame([{'estrategia':'Meta-LightGBM',
                                                  'AUC-ROC':auc_roc_gbm, 'AUC-PR':auc_pr_gbm,
                                                  'F1':f1_gbm}])], ignore_index=True)
print('COMPARACION DE ENSEMBLES:')
print(summary.to_string(index=False))

best_strat = summary.sort_values('AUC-PR', ascending=False).iloc[0]['estrategia']
print(f'\n  GANADOR: {best_strat}')

# Mapear ganador a sus scores y threshold
score_map = {
    'Mean blending':     (s_test_mean, thr_m),
    'Weighted blending': (s_test_wb,   best_blend['thr']),
    'Meta-LR':           (s_test_lr,   thr_lr),
}
if HAS_LGBM:
    score_map['Meta-LightGBM'] = (s_test_gbm, thr_gbm)

scores_test_final, best_thr_final = score_map[best_strat]


COMPARACION DE ENSEMBLES:
       estrategia  AUC-ROC   AUC-PR       F1
    Mean blending 0.890058 0.593840 0.635374
Weighted blending 0.927394 0.763560 0.731673
          Meta-LR 0.933945 0.792253 0.755604
    Meta-LightGBM 0.970983 0.920450 0.842182

  GANADOR: Meta-LightGBM


## 10. Postproceso temporal + Eval smoothed

In [ ]:
def temporal_vote_vectorized(y_pred, W, K):
    '''Filtro de votacion causal. O(n) con cumsum.'''
    cum = np.concatenate([[0], np.cumsum(y_pred)])
    n = len(y_pred)
    ends = np.arange(1, n + 1)
    starts = np.maximum(0, ends - W)
    votes = cum[ends] - cum[starts]
    return (votes >= K).astype(int)


def grid_search_wk(scores_val, y_val, best_thr, W_grid=(3,5,7,9,11,15,21,30)):
    '''Busca (W,K) que maximiza F1 en val.'''
    y_pred_val = (scores_val > best_thr).astype(int)
    best_f1, best_W, best_K = 0, 3, 2
    for W in W_grid:
        for K in range(2, W + 1):
            y_sm = temporal_vote_vectorized(y_pred_val, W, K)
            f1 = f1_score(y_val, y_sm)
            if f1 > best_f1:
                best_f1, best_W, best_K = f1, W, K
    return best_W, best_K, best_f1

# Para postproceso usamos los scores en val del estrategia ganadora
score_val_map = {
    'Mean blending':     s_val_mean,
    'Weighted blending': S_val @ best_blend['weights'],
    'Meta-LR':           meta_lr.predict_proba(S_val)[:, 1],
}
if HAS_LGBM:
    score_val_map['Meta-LightGBM'] = gbm_meta.predict(S_val)
scores_val_final = score_val_map[best_strat]

best_W, best_K, _ = grid_search_wk(scores_val_final, y_val, best_thr_final)
print(f'Filtro temporal optimo (ensemble): W={best_W}, K={best_K}')

y_pred_raw = (scores_test_final > best_thr_final).astype(int)
y_pred_smooth = temporal_vote_vectorized(y_pred_raw, best_W, best_K)

f1_r  = f1_score(y_test, y_pred_raw)
f2_r  = fbeta_score(y_test, y_pred_raw, beta=2)
prec_r = precision_score(y_test, y_pred_raw)
rec_r  = recall_score(y_test, y_pred_raw)
auc_roc = roc_auc_score(y_test, scores_test_final)
auc_pr  = average_precision_score(y_test, scores_test_final)

f1_s  = f1_score(y_test, y_pred_smooth)
f2_s  = fbeta_score(y_test, y_pred_smooth, beta=2)
prec_s = precision_score(y_test, y_pred_smooth)
rec_s  = recall_score(y_test, y_pred_smooth)

print('='*60); print(f'ENSEMBLE ({best_strat}) — RAW vs SMOOTHED'); print('='*60)
print(f'  RAW:      F1={f1_r:.4f} F2={f2_r:.4f} P={prec_r:.4f} R={rec_r:.4f}')
print(f'  SMOOTHED: F1={f1_s:.4f} F2={f2_s:.4f} P={prec_s:.4f} R={rec_s:.4f}')
print(f'  AUC-ROC={auc_roc:.4f}  AUC-PR={auc_pr:.4f}')


# ── ITER PALANCA A: Joint search (threshold × W × K) sobre val ──
prc_pr_, prc_rc_, prc_thr_ = precision_recall_curve(y_val, scores_val_final)
prc_thr_ = prc_thr_[prc_thr_ > 0]
if len(prc_thr_) < 5:
    thr_candidates = np.linspace(scores_val_final.min(), scores_val_final.max(), 40)
else:
    thr_candidates = np.unique(np.quantile(prc_thr_, np.linspace(0.30, 0.999, 80)))
W_grid_joint = [3, 5, 7, 9, 11, 15, 21, 31, 45, 60]
best_joint_f1 = -1.0
best_joint = (best_thr_final_final, best_W, best_K)
for thr in thr_candidates:
    preds_v = (scores_val_final > thr).astype(int)
    if preds_v.sum() in (0, len(preds_v)):
        continue
    for W in W_grid_joint:
        for K in range(2, W+1):
            y_sm = temporal_vote_vectorized(preds_v, W, K)
            if y_sm.sum() == 0:
                continue
            f1 = f1_score(y_val, y_sm)
            if f1 > best_joint_f1:
                best_joint_f1 = f1
                best_joint = (thr, W, K)
preds_v0 = (scores_val_final > best_thr_final_final).astype(int)
y_sm0 = temporal_vote_vectorized(preds_v0, best_W, best_K)
f1_val_orig = f1_score(y_val, y_sm0)
print(f'\nJOINT SEARCH NB10:')
print(f'  best joint val F1: {best_joint_f1:.4f}  thr={best_joint[0]:.6f} W={best_joint[1]} K={best_joint[2]}')
print(f'  orig val F1:       {f1_val_orig:.4f}  thr={best_thr_final_final:.6f} W={best_W} K={best_K}')
if best_joint_f1 > f1_val_orig:
    best_thr_final_final, best_W, best_K = best_joint
    y_pred_raw = (scores_test_final > best_thr_final_final).astype(int)
    print('  -> adoptamos joint')
else:
    print('  -> mantenemos previo')

# Re-aplicar
y_pred_smooth = temporal_vote_vectorized(y_pred_raw, best_W, best_K)
f1_s = f1_score(y_test, y_pred_smooth)
f2_s = fbeta_score(y_test, y_pred_smooth, beta=2)
prec_s = precision_score(y_test, y_pred_smooth)
rec_s  = recall_score(y_test, y_pred_smooth)
f1_r  = f1_score(y_test, y_pred_raw)
f2_r  = fbeta_score(y_test, y_pred_raw, beta=2)
prec_r = precision_score(y_test, y_pred_raw)
rec_r  = recall_score(y_test, y_pred_raw)
print(f'TEST tras joint: F1_smoothed={f1_s:.4f}  P={prec_s:.4f}  R={rec_s:.4f}')


## 11. Recall por tipo + Latencia + Guardado

In [15]:
def recall_per_type_severity(df_preds, score_col, threshold,
                              attack_types=ATTACK_TYPES,
                              severities=('low','medium','high')):
    '''Recall por (tipo, severidad).'''
    rows = []
    for atype in attack_types:
        row = {'type': atype}
        for sev in severities:
            mask = (df_preds['attack_type']==atype) & (df_preds['severity']==sev)
            if mask.sum() == 0:
                row[sev] = np.nan
            else:
                y_true_sub  = df_preds.loc[mask, 'label'].values
                y_score_sub = df_preds.loc[mask, score_col].values
                y_pred_sub  = (y_score_sub > threshold).astype(int)
                row[sev] = recall_score(y_true_sub, y_pred_sub, zero_division=0)
        rows.append(row)
    out = pd.DataFrame(rows).set_index('type')
    print('Recall por tipo x severidad (smoothed):')
    print(out.round(3))
    return out
def compute_detection_latency(df_preds, score_col, threshold):
    '''Latencia mediana por episodio (minutos).'''
    results = []
    attacked = df_preds[df_preds['episode_id'] >= 0]
    for ep_id, group in attacked.groupby('episode_id'):
        group = group.sort_index()
        detections = group[group[score_col] > threshold]
        ep_start = group.index[0]
        results.append({
            'episode_id': ep_id,
            'type': group['attack_type'].iloc[0],
            'severity': group['severity'].iloc[0],
            'duration_min': len(group),
            'latency_min': ((detections.index[0] - ep_start).total_seconds() / 60
                            if len(detections) > 0 else np.nan),
            'detected': len(detections) > 0
        })
    return pd.DataFrame(results)


def print_latency_summary(lat_df):
    det_rate = lat_df['detected'].mean()
    detected = lat_df[lat_df['detected']]
    if len(detected) > 0:
        med_lat = detected['latency_min'].median()
        p90_lat = detected['latency_min'].quantile(0.9)
    else:
        med_lat = p90_lat = np.nan
    print('=' * 65)
    print('LATENCIA DE DETECCION (smoothed)')
    print('=' * 65)
    print(f'  Detection rate: {det_rate*100:.2f}%')
    print(f'  Mediana:        {med_lat:.1f} min')
    print(f'  P90:            {p90_lat:.1f} min')
    print()
    print('  Por tipo de ataque:')
    for atype, g in lat_df.groupby('type'):
        gd = g[g['detected']]
        print(f'    {atype:18s} n={len(g):3d}  det={g["detected"].mean()*100:5.1f}%  '
              f'med_lat={gd["latency_min"].median() if len(gd)>0 else float("nan"):5.1f}min')
    return det_rate, med_lat, p90_lat

df_preds_test = df_test[['attack_type','severity','episode_id','label']].copy()
df_preds_test['score']   = scores_test_final
df_preds_test['pred_sm'] = y_pred_smooth

def recall_per_type_severity_sm(df_preds, pred_col,
                                attack_types=ATTACK_TYPES,
                                severities=('low','medium','high')):
    rows = []
    for atype in attack_types:
        row = {'type': atype}
        for sev in severities:
            mask = (df_preds['attack_type']==atype) & (df_preds['severity']==sev)
            if mask.sum() == 0:
                row[sev] = np.nan
            else:
                row[sev] = recall_score(df_preds.loc[mask,'label'].values,
                                         df_preds.loc[mask,pred_col].values, zero_division=0)
        rows.append(row)
    out = pd.DataFrame(rows).set_index('type')
    print('Recall por tipo x severidad (smoothed):')
    print(out.round(3))
    return out

recall_table = recall_per_type_severity_sm(df_preds_test, 'pred_sm')
lat = compute_detection_latency(df_preds_test, 'score', best_thr_final)
det_rate, med_lat, p90_lat = print_latency_summary(lat)

df_preds_test.to_csv(os.path.join(DATA_DIR, 'predictions_stacking.csv'))
lat.to_csv(os.path.join(DATA_DIR, 'latency_stacking.csv'), index=False)

# Tamaño total del ensemble = suma de los detectores que lo componen
total_size_kb = 0
for name in models_in_stack:
    cfg = load_pipeline_config_for(name)
    if cfg is None: continue
    mdl_fn = cfg[1]
    p = os.path.join(DATA_DIR, mdl_fn)
    if os.path.exists(p):
        total_size_kb += os.path.getsize(p)/1e3

metrics = {
    'model': f'Stacking ({best_strat}) + Temporal Voting',
    'detectors': models_in_stack,
    'best_strategy': best_strat,
    'raw_f1':        round(float(f1_r), 4),
    'raw_f2':        round(float(f2_r), 4),
    'raw_precision': round(float(prec_r), 4),
    'raw_recall':    round(float(rec_r), 4),
    'f1':            round(float(f1_s), 4),
    'f2':            round(float(f2_s), 4),
    'precision':     round(float(prec_s), 4),
    'recall':        round(float(rec_s), 4),
    'auc_roc':       round(float(auc_roc), 4),
    'auc_pr':        round(float(auc_pr), 4),
    'threshold':     float(best_thr_final),
    'temporal_W':    int(best_W),
    'temporal_K':    int(best_K),
    'total_size_kb': round(total_size_kb, 1),
    'detection_rate':round(float(det_rate), 4),
    'median_latency_min': float(med_lat) if not np.isnan(med_lat) else None,
    'meta_weights': dict(zip(models_in_stack, meta_lr.coef_[0].round(3).tolist())) if best_strat == 'Meta-LR' else None,
    'summary_table': summary.round(4).to_dict(orient='records'),
}
with open(os.path.join(DATA_DIR, 'metrics_stacking.json'), 'w') as f:
    json.dump(metrics, f, indent=2)

# Persistir el meta-LR
joblib.dump(meta_lr, os.path.join(DATA_DIR, 'meta_lr_stacking.pkl'))
print('Guardado ensemble completo en data/.')


Recall por tipo x severidad (smoothed):
                 low  medium   high
type                               
scaling        0.790   0.978  0.987
offset         0.794   0.983  0.988
noise          0.859   0.976  0.984
ramp           0.589   0.812  0.885
step           0.569   0.678  0.658
replay         0.667   0.862  0.899
voltage_spoof  0.908   0.990  0.991
LATENCIA DE DETECCION (smoothed)
  Detection rate: 98.45%
  Mediana:        0.0 min
  P90:            29.0 min

  Por tipo de ataque:
    noise              n=120  det=100.0%  med_lat=  0.0min
    offset             n=120  det= 97.5%  med_lat=  0.0min
    ramp               n=120  det= 97.5%  med_lat= 12.0min
    replay             n=120  det=100.0%  med_lat=  0.0min
    scaling            n=120  det= 97.5%  med_lat=  0.0min
    step               n=120  det= 97.5%  med_lat= 23.0min
    voltage_spoof      n=120  det= 99.2%  med_lat=  0.0min
Guardado ensemble completo en data/.


## 12. Resumen

In [16]:
print('='*65)
print(f'NB10 — STACKING ({best_strat}): RESUMEN')
print('='*65)
print(f'  Detectores: {", ".join(models_in_stack)}')
print(f'  AUC-ROC:    {auc_roc:.4f}')
print(f'  AUC-PR:     {auc_pr:.4f}')
print(f'  F1 raw:     {f1_r:.4f} -> smoothed: {f1_s:.4f}')
print(f'  Latencia mediana: {med_lat:.1f} min')
print(f'  Detection rate:   {det_rate*100:.2f}%')
print(f'  Tamano total ensemble: {total_size_kb:.0f} KB')
print()
print('La ganancia esperada del stacking es de 1-3 puntos de F1 sobre el')
print('mejor detector individual, gracias a la complementariedad de errores.')


NB10 — STACKING (Meta-LightGBM): RESUMEN
  Detectores: if, cnn_ae, transformer
  AUC-ROC:    0.9710
  AUC-PR:     0.9204
  F1 raw:     0.8422 -> smoothed: 0.8428
  Latencia mediana: 0.0 min
  Detection rate:   98.45%
  Tamano total ensemble: 4795 KB

La ganancia esperada del stacking es de 1-3 puntos de F1 sobre el
mejor detector individual, gracias a la complementariedad de errores.
